In [ ]:
!git clone https://github.com/AMAAI-Lab/text2midi
!pip install -r text2midi/requirements.txt
!pip install --upgrade torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install music21 librosa pretty_midi laion-clap scipy numpy

fatal: destination path 'text2midi' already exists and is not an empty directory.
  Using cached numpy-2.0.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
  Using cached sympy-1.13.1-py3-none-any.whl.metadata (12 kB)
  Using cached torch-2.5.1-cp311-cp311-manylinux1_x86_64.whl.metadata (28 kB)
  Using cached triton-3.1.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.3 kB)
Using cached numpy-2.0.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (19.5 MB)
Using cached sympy-1.13.1-py3-none-any.whl (6.2 MB)
Using cached torch-2.5.1-cp311-cp311-manylinux1_x86_64.whl (906.5 MB)
Using cached triton-3.1.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (209.5 MB)
  Attempting uninstall: triton
    Found existing installation: triton 3.3.1
    Uninstalling triton-3.3.1:
      Successfully uninstalled triton-3.3.1
  Attempting uninstall: sympy
    Found existing installation: sympy 1.13.3
    Uninstalling sympy-1.13.3:

Looking in indexes: https://download.pytorch.org/whl/cu118
  Using cached https://download.pytorch.org/whl/cu118/torch-2.7.1%2Bcu118-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (28 kB)
  Using cached https://download.pytorch.org/whl/sympy-1.13.3-py3-none-any.whl.metadata (12 kB)
  Using cached https://download.pytorch.org/whl/triton-3.3.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (1.5 kB)
Using cached https://download.pytorch.org/whl/cu118/torch-2.7.1%2Bcu118-cp311-cp311-manylinux_2_28_x86_64.whl (905.3 MB)
Using cached https://download.pytorch.org/whl/triton-3.3.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (155.7 MB)
Using cached https://download.pytorch.org/whl/sympy-1.13.3-py3-none-any.whl (6.2 MB)
  Attempting uninstall: triton
    Found existing installation: triton 3.1.0
    Uninstalling triton-3.1.0:
      Successfully uninstalled triton-3.1.0
  Attempting uninstall: sympy
    Found existing installation: sympy 1.13.1
    Uninstalli

In [ ]:
import os
import pickle
import torch
import torch.nn as nn
from transformers import T5Tokenizer, AutoTokenizer
from huggingface_hub import hf_hub_download
import subprocess
import sys
import scipy
import soundfile as sf
import numpy as np

In [ ]:
os.chdir('text2midi')
!mkdir -p text2midi/model
!touch text2midi/model/__init__.py

In [ ]:
# Cell 1: Import Libraries and Setup
import os
import pickle
import torch
import torch.nn as nn
from transformers import T5Tokenizer, AutoTokenizer
from model.transformer_model import Transformer # Now, the import should work
from huggingface_hub import hf_hub_download
import music21
import numpy as np
from scipy.spatial.distance import cosine
import librosa;
import pretty_midi
from typing import List, Dict, Tuple
import json
import scipy
from scipy.spatial.distance import cosine
import random
from datasets import load_dataset
import tempfile
import requests

# CLAP imports - you may need to install these
try:
  import laion_clap
except ImportError:
    print("Warning: laion_clap not installed. Install with: pip install laion-clap")
    laion_clap = None
!pip install laion-clap

/usr/local/lib/python3.11/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.72k [00:00<?, ?B/s]

In [ ]:
def safe_float_conversion(value):
    """Safely convert numpy scalars or other types to Python float"""
    if isinstance(value, (np.floating, np.integer)):
        return float(value)
    elif isinstance(value, (int, float)):
        return float(value)
    elif hasattr(value, 'item'):  # numpy scalar
        return float(value.item())
    else:
        try:
            return float(value)
        except:
            return 0.0


# Cell 4: Define ObjectiveEvaluator Class
class ObjectiveEvaluator:
    def __init__(self):
        # Initialize CLAP model if available
        self.clap_model = None
        if laion_clap is not None:
            try:
                self.clap_model = laion_clap.CLAP_Module(enable_fusion=False)
                self.clap_model.load_ckpt() # Load the default checkpoint
                print("CLAP model loaded successfully")
            except Exception as e:
                print(f"Failed to load CLAP model: {e}")

        # Tempo bins as defined in the paper
        self.tempo_bins = [40, 60, 70, 90, 110, 140, 160, 210]

    def evaluate_single_sample(self, text: str, generated_midi_path: str, ground_truth_midi_path: str) -> Dict:
        """Evaluate a single sample with all metrics - returns Python native types"""
        results = {}

        # Compression Ratio
        results['compression_ratio_generated'] = safe_float_conversion(
            self.compute_compression_ratio(generated_midi_path)
        )
        results['compression_ratio_ground_truth'] = safe_float_conversion(
            self.compute_compression_ratio(ground_truth_midi_path)
        )

        # CLAP Similarity - this is where the error was occurring
        clap_sim = self.compute_clap_similarity(text, generated_midi_path)
        results['clap_similarity'] = safe_float_conversion(clap_sim)

        # Feature-wise comparison
        gen_tempo = self.extract_tempo(generated_midi_path)
        gt_tempo = self.extract_tempo(ground_truth_midi_path)
        results['generated_tempo'] = safe_float_conversion(gen_tempo)
        results['ground_truth_tempo'] = safe_float_conversion(gt_tempo)

        tb, tbt = self.compute_tempo_bin_accuracy(gen_tempo, gt_tempo)
        results['tempo_bin_match'] = bool(tb)  # Ensure boolean
        results['tempo_bin_tolerance_match'] = bool(tbt)  # Ensure boolean

        gen_key = self.extract_key(generated_midi_path)
        gt_key = self.extract_key(ground_truth_midi_path)
        results['generated_key'] = str(gen_key)  # Ensure string
        results['ground_truth_key'] = str(gt_key)  # Ensure string

        ck, ckd = self.compute_key_accuracy(gen_key, gt_key)
        results['correct_key'] = bool(ck)  # Ensure boolean
        results['correct_key_duplicates'] = bool(ckd)  # Ensure boolean

        return results

    def midi_to_audio(self, midi_file_path: str, sample_rate: int = 22050) -> np.ndarray:
        """Convert MIDI file to audio using pretty_midi and fluidsynth"""
        try:
            # Load MIDI file
            midi_data = pretty_midi.PrettyMIDI(midi_file_path)

            # Check if MIDI has any instruments
            if len(midi_data.instruments) == 0:
                print("Warning: MIDI file has no instruments, returning silence")
                return np.zeros(sample_rate * 2, dtype=np.float32)  # 2 seconds of silence

            # Synthesize audio
            audio = midi_data.synthesize(fs=sample_rate)

            # **FIX: Ensure we always return a proper numpy array**
            if not isinstance(audio, np.ndarray):
                audio = np.array(audio, dtype=np.float32)
            else:
                audio = audio.astype(np.float32)

            # Handle edge cases - ensure we always return an array, never a scalar
            if audio.size == 0:
                print("Warning: MIDI synthesis produced empty audio, returning silence")
                return np.zeros(sample_rate * 2, dtype=np.float32)
            elif audio.size == 1:
                print("Warning: MIDI synthesis produced single sample, extending to array")
                # Create an array by repeating the single value
                single_value = float(audio.item()) if hasattr(audio, 'item') else float(audio)
                return np.full(sample_rate * 2, single_value, dtype=np.float32)
            elif np.isscalar(audio):
                # Handle scalar case explicitly
                print("Warning: MIDI synthesis returned scalar, converting to array")
                return np.full(sample_rate * 2, float(audio), dtype=np.float32)

            # Ensure it's 1D
            if audio.ndim > 1:
                audio = audio.flatten()
            elif audio.ndim == 0:  # Handle 0-dimensional arrays
                single_value = float(audio.item()) if hasattr(audio, 'item') else float(audio)
                return np.full(sample_rate * 2, single_value, dtype=np.float32)

            return audio

        except Exception as e:
            print(f"Error converting MIDI to audio: {e}")
            # Return silence instead of empty array
            return np.zeros(22050 * 2, dtype=np.float32)  # 2 seconds of silence at 22050 Hz

    def compute_clap_similarity(self, text: str, midi_file_path: str) -> float:
        """Compute CLAP similarity between text and MIDI audio"""
        if self.clap_model is None:
            print("CLAP model not available")
            return 0.0

        try:
            # Convert MIDI to audio
            audio = self.midi_to_audio(midi_file_path)

            # **FIX: Additional validation to ensure we have a proper array**
            if not isinstance(audio, np.ndarray):
                print(f"Error: midi_to_audio returned {type(audio)} instead of np.ndarray")
                return 0.0

            if audio.ndim == 0 or np.isscalar(audio):
                print("Error: audio is scalar, converting to array")
                audio = np.full(22050, float(audio) if np.isscalar(audio) else float(audio.item()), dtype=np.float32)

            if audio.size == 0:
                print("Error: audio array is empty")
                return 0.0

            # Ensure minimum length for CLAP processing
            if len(audio) < 1000:
                if len(audio) == 0:
                    audio = np.zeros(22050, dtype=np.float32)  # 1 second of silence
                else:
                    repeat_factor = max(1, 1000 // len(audio) + 1)
                    audio = np.tile(audio, repeat_factor)[:22050]  # Ensure reasonable length

            print(f"Final audio validation - type: {type(audio)}, shape: {audio.shape}, dtype: {audio.dtype}")

            # Get embeddings
            text_embed = self.clap_model.get_text_embedding([text], use_tensor=False)

            # **FIX: CLAP library workaround - save audio to temporary file**
            # The CLAP library has internal issues with numpy arrays, so we'll use file-based approach
            import tempfile
            import scipy.io.wavfile as wavfile

            try:
                # Create temporary wav file
                with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp_file:
                    temp_path = tmp_file.name

                # Normalize audio to 16-bit range and save as WAV
                audio_normalized = audio.astype(np.float64)
                # Ensure audio is in proper range [-1, 1]
                if np.max(np.abs(audio_normalized)) > 0:
                    audio_normalized = audio_normalized / np.max(np.abs(audio_normalized))

                # Convert to 16-bit integer
                audio_16bit = (audio_normalized * 32767).astype(np.int16)

                # Save as WAV file
                wavfile.write(temp_path, 22050, audio_16bit)

                print(f"Saved temporary audio file: {temp_path}")

                # Try file-based CLAP embedding methods
                audio_embed = None

                # Method 1: Try get_audio_embedding_from_filelist if available
                if hasattr(self.clap_model, 'get_audio_embedding_from_filelist'):
                    try:
                        audio_embed = self.clap_model.get_audio_embedding_from_filelist([temp_path], use_tensor=False)
                        print("Successfully used get_audio_embedding_from_filelist")
                    except Exception as e:
                        print(f"get_audio_embedding_from_filelist failed: {e}")

                # Method 2: Try get_audio_embedding if available
                if audio_embed is None and hasattr(self.clap_model, 'get_audio_embedding'):
                    try:
                        audio_embed = self.clap_model.get_audio_embedding([temp_path], use_tensor=False)
                        print("Successfully used get_audio_embedding")
                    except Exception as e:
                        print(f"get_audio_embedding failed: {e}")

                # Method 3: Load the file back and try data method with fresh array
                if audio_embed is None:
                    try:
                        # Load the audio file we just saved
                        sample_rate, audio_from_file = wavfile.read(temp_path)

                        # Convert back to float
                        audio_from_file = audio_from_file.astype(np.float64) / 32767.0

                        # Ensure it's a proper numpy array
                        audio_from_file = np.asarray(audio_from_file, dtype=np.float64)

                        print(f"Loaded audio from file - type: {type(audio_from_file)}, shape: {audio_from_file.shape}, dtype: {audio_from_file.dtype}")

                        audio_embed = self.clap_model.get_audio_embedding_from_data(x=audio_from_file, use_tensor=False)
                        print("Successfully used get_audio_embedding_from_data with file-loaded audio")

                    except Exception as e:
                        print(f"File-based data method failed: {e}")

                # Clean up temporary file
                import os
                if os.path.exists(temp_path):
                    os.unlink(temp_path)

                if audio_embed is None:
                    print("All CLAP embedding methods failed, returning 0.0")
                    return 0.0

            except Exception as e:
                print(f"Error in file-based CLAP approach: {e}")
                return 0.0

            # Unwrap embeddings if they come as lists
            if isinstance(text_embed, list):
                text_embed = text_embed[0]
            if isinstance(audio_embed, list):
                audio_embed = audio_embed[0]

            # Convert to numpy arrays if they aren't already
            if not isinstance(text_embed, np.ndarray):
                text_embed = np.array(text_embed)
            if not isinstance(audio_embed, np.ndarray):
                audio_embed = np.array(audio_embed)

            # Ensure they are proper numpy arrays with float64 dtype
            text_embed = np.asarray(text_embed, dtype=np.float64)
            audio_embed = np.asarray(audio_embed, dtype=np.float64)

            # Handle different dimensionalities
            if text_embed.ndim > 1:
                text_embed = text_embed.flatten()
            if audio_embed.ndim > 1:
                audio_embed = audio_embed.flatten()

            # Final validation
            if not isinstance(text_embed, np.ndarray) or not isinstance(audio_embed, np.ndarray):
                print(f"CLAP embeddings are not ndarray: text {type(text_embed)}, audio {type(audio_embed)}")
                return 0.0

            if text_embed.size == 0 or audio_embed.size == 0:
                print("CLAP embeddings are empty")
                return 0.0

            if text_embed.shape != audio_embed.shape:
                print(f"CLAP embedding shape mismatch: text {text_embed.shape}, audio {audio_embed.shape}")
                return 0.0

            # Compute cosine similarity using manual calculation
            # Normalize embeddings
            text_embed_norm = text_embed / (np.linalg.norm(text_embed) + 1e-8)
            audio_embed_norm = audio_embed / (np.linalg.norm(audio_embed) + 1e-8)

            # Compute cosine similarity manually
            similarity = np.dot(text_embed_norm, audio_embed_norm)

            # Ensure we return a Python float, not numpy scalar
            if hasattr(similarity, 'item'):
                return float(similarity.item())
            else:
                return float(similarity)

        except Exception as e:
            print(f"Error computing CLAP similarity: {e}")
            import traceback
            traceback.print_exc()  # This will help debug the exact error
            return 0.0

    def compute_compression_ratio(self, midi_file_path: str) -> float:
        """
        Compute compression ratio using COSIATEC algorithm approximation
        Note: This is a simplified version. For exact COSIATEC, you'd need the original implementation
        """
        try:
            # Load MIDI file
            score = music21.converter.parse(midi_file_path)

            # Extract notes and their properties
            notes = []
            for element in score.recurse().notes:
                if hasattr(element, 'pitch'):  # Single note
                    notes.append({
                        'pitch': element.pitch.midi,
                        'offset': float(element.offset),
                        'duration': float(element.duration.quarterLength)
                    })
                elif hasattr(element, 'pitches'):  # Chord
                    for pitch in element.pitches:
                        notes.append({
                            'pitch': pitch.midi,
                            'offset': float(element.offset),
                            'duration': float(element.duration.quarterLength)
                        })

            if len(notes) == 0:
                return 0.0

            # Simple pattern detection based on repeated pitch sequences
            # This is a simplified approximation of COSIATEC
            patterns = {}
            window_size = 4  # Look for patterns of 4 notes

            for i in range(len(notes) - window_size + 1):
                pattern = tuple(note['pitch'] for note in notes[i:i+window_size])
                if pattern in patterns:
                    patterns[pattern] += 1
                else:
                    patterns[pattern] = 1

            # Calculate compression ratio
            total_notes = len(notes)
            repeated_notes = sum(count * len(pattern) for pattern, count in patterns.items() if count > 1)
            compression_ratio = repeated_notes / total_notes if total_notes > 0 else 0.0

            return compression_ratio

        except Exception as e:
            print(f"Error computing compression ratio: {e}")
            return 0.0

    def extract_tempo(self, midi_file_path: str) -> float:
        """Extract tempo from MIDI file"""
        try:
            score = music21.converter.parse(midi_file_path)
            tempos = score.metronomeMarkBoundaries()
            if tempos:
                return float(tempos[0][2].number)
            else:
                # Fallback: analyze note density
                notes = list(score.recurse().notes)
                if len(notes) > 1:
                    duration = float(score.duration.quarterLength)
                    note_density = len(notes) / duration
                    estimated_tempo = note_density * 60  # Rough estimation
                    return max(60, min(200, estimated_tempo))  # Clamp to reasonable range
                return 120  # Default tempo
        except Exception as e:
            print(f"Error extracting tempo: {e}")
            return 120

    def extract_key(self, midi_file_path: str) -> str:
        """Extract key from MIDI file"""
        try:
            score = music21.converter.parse(midi_file_path)
            key = score.analyze('key')
            return str(key)
        except Exception as e:
            print(f"Error extracting key: {e}")
            return "C major"

    def get_tempo_bin(self, tempo: float) -> int:
        """Get the tempo bin index for a given tempo"""
        for i, bin_edge in enumerate(self.tempo_bins):
            if tempo < bin_edge:
                return i
        return len(self.tempo_bins)

    def compute_tempo_bin_accuracy(self, generated_tempo: float, ground_truth_tempo: float) -> tuple:
        """Compute TB and TBT metrics"""
        gen_bin = self.get_tempo_bin(generated_tempo)
        gt_bin = self.get_tempo_bin(ground_truth_tempo)

        # TB: exact bin match
        tb = (gen_bin == gt_bin)

        # TBT: exact bin or neighboring bin
        tbt = abs(gen_bin - gt_bin) <= 1

        return tb, tbt

    def normalize_key(self, key_str: str) -> str:
        """Normalize key string for comparison"""
        try:
            key = music21.key.Key(key_str)
            return str(key).lower().strip()
        except:
            return key_str.lower().strip()

    def are_relative_keys(self, key1: str, key2: str) -> bool:
        """Check if two keys are relative (major and its relative minor)"""
        try:
            k1 = music21.key.Key(key1)
            k2 = music21.key.Key(key2)

            if k1.mode == 'major' and k2.mode == 'minor':
                return k1.relative.name == k2.name
            elif k1.mode == 'minor' and k2.mode == 'major':
                return k1.relative.name == k2.name
            else:
                return k1.name == k2.name
        except:
            return False

    def compute_key_accuracy(self, generated_key: str, ground_truth_key: str) -> tuple:
        """Compute CK and CKD metrics"""
        # CK: exact key match
        ck = self.normalize_key(generated_key) == self.normalize_key(ground_truth_key)

        # CKD: exact key or relative key match
        ckd = ck or self.are_relative_keys(generated_key, ground_truth_key)

        return ck, ckd

In [ ]:
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

from google.colab import drive
drive.mount('/content/drive')
model_path = '/content/drive/MyDrive/pytorch_model.bin'
vocab_path = '/content/drive/MyDrive/vocab_remi.pkl'

import pickle

with open(vocab_path, 'rb') as f:
    r_tokenizer = pickle.load(f)
vocab_size = len(r_tokenizer)
import torch

# Assuming you have defined or imported YourModelClass somewhere
model = Transformer(vocab_size, 768, 8, 2048, 18, 1024, False, 8, device=device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")

# CLAP imports - you may need to install these
try:
  import laion_clap
except ImportError:
    print("Warning: laion_clap not installed. Install with: pip install laion-clap")
    laion_clap = None


Mounted at /content/drive


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [ ]:
#vol 2
from datasets import load_dataset
from typing import List, Dict
import os
import random
import json
import numpy as np

def load_midicaps_samples(num_samples: int = 100, split: str = "train", midi_dir: str = None) -> List[Dict]:
    """
    Load captions from MidiCaps (Hugging Face) and match with local MIDI files from lmd_full
    """
    if midi_dir is None:
        print("⚠️ You must provide the path to your local lmd_full directory.")
        return []

    print(f"Using MIDI directory: {midi_dir}")
    print("Loading MidiCaps dataset from Hugging Face...")

    try:
        dataset = load_dataset("amaai-lab/MidiCaps", split=split)
        print(f"✅ Loaded {len(dataset)} samples from '{split}' split.")

        if num_samples > len(dataset):
            print("⚠️ Requested more samples than available. Using all.")
            num_samples = len(dataset)

        selected_indices = random.sample(range(len(dataset)), num_samples)
        samples = []

        for i, idx in enumerate(selected_indices):
            sample = dataset[idx]
            caption = sample['caption']
            midi_rel_path = sample['location']  # e.g. 'lmd_full/2/abc123.mid'

            # Strip leading 'lmd_full/' if present
            if midi_rel_path.startswith("lmd_full/"):
                midi_rel_path = midi_rel_path[len("lmd_full/"):]

            midi_abs_path = os.path.join(midi_dir, midi_rel_path)

            if not os.path.exists(midi_abs_path):
                print(f"❌ Missing MIDI file: {midi_abs_path}")
                continue

            samples.append({
                'text': caption,
                'midi_path': midi_abs_path,
                'original_index': idx,
                'sample_id': i
            })

        print(f"✅ Collected {len(samples)} valid samples with existing local MIDI files.")
        return samples

    except Exception as e:
        print(f"❌ Error loading MidiCaps dataset: {e}")
        return []


def batch_evaluate_midicaps(evaluator, num_samples: int = 100, midi_dir: str = None):
    test_samples = load_midicaps_samples(num_samples=num_samples, midi_dir=midi_dir)
    if not test_samples:
        print("❌ No valid test samples found.")
        return []

    all_results = []

    for i, sample in enumerate(test_samples):
        print(f"\nProcessing sample {i+1}/{len(test_samples)} (Original index: {sample['original_index']})")

        text = sample['text']
        gt_midi_path = sample['midi_path']
        print(f"Caption: {text[:100]}...")
        print(f"Ground truth MIDI: {gt_midi_path}")

        if not os.path.exists(gt_midi_path):
            print(f"❌ MIDI file not found: {gt_midi_path}")
            continue

        try:
            # Generate MIDI
            inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True)
            input_ids = nn.utils.rnn.pad_sequence(inputs.input_ids, batch_first=True, padding_value=0).to(device)
            attention_mask = nn.utils.rnn.pad_sequence(inputs.attention_mask, batch_first=True, padding_value=0).to(device)

            output = model.generate(input_ids, attention_mask, max_len=2000, temperature=1.0)
            output_list = output[0].tolist()
            generated_midi = r_tokenizer.decode(output_list)

            # Save generated MIDI
            generated_path = f"generated_{i}.mid"
            generated_midi.dump_midi(generated_path)

            # Evaluate - this is where the error occurs
            results = evaluator.evaluate_single_sample(text, generated_path, gt_midi_path)

            # **FIX: Convert all numeric values to Python floats**
            results_cleaned = {}
            for key, value in results.items():
                if isinstance(value, (str, bool)):
                    results_cleaned[key] = value
                else:
                    results_cleaned[key] = safe_float_conversion(value)

            results_cleaned.update({
                'sample_id': i,
                'original_index': sample['original_index'],
                'caption': text,
                'ground_truth_path': gt_midi_path
            })
            all_results.append(results_cleaned)

            os.remove(generated_path)

        except Exception as e:
            print(f"❌ Error evaluating sample {i}: {e}")
            import traceback
            traceback.print_exc()  # This will help debug
            continue

    if not all_results:
        print("⚠️ No samples were successfully evaluated.")
        return []

    # Print aggregate stats with safe conversion
    print(f"\n=== AGGREGATE RESULTS ({len(all_results)} samples) ===")

    # Safe aggregation with explicit conversion
    compression_gen = [safe_float_conversion(r['compression_ratio_generated']) for r in all_results]
    compression_gt = [safe_float_conversion(r['compression_ratio_ground_truth']) for r in all_results]

    print(f"Average Compression Ratio (Gen): {np.mean(compression_gen):.4f}")
    print(f"Average Compression Ratio (GT): {np.mean(compression_gt):.4f}")

    # CLAP similarities with safe conversion
    clap_sims = [safe_float_conversion(r['clap_similarity']) for r in all_results
                 if safe_float_conversion(r['clap_similarity']) > 0]

    if clap_sims:
        print(f"Average CLAP Similarity: {np.mean(clap_sims):.4f} ± {np.std(clap_sims):.4f}")
    else:
        print("No valid CLAP similarities computed")

    # Convert boolean results to float for aggregation
    tb_acc = np.mean([float(r['tempo_bin_match']) for r in all_results])
    tbt_acc = np.mean([float(r['tempo_bin_tolerance_match']) for r in all_results])
    ck_acc = np.mean([float(r['correct_key']) for r in all_results])
    ckd_acc = np.mean([float(r['correct_key_duplicates']) for r in all_results])

    print(f"Tempo Bin Accuracy (TB): {tb_acc:.4f} | TBT: {tbt_acc:.4f}")
    print(f"Correct Key Accuracy (CK): {ck_acc:.4f} | CKD: {ckd_acc:.4f}")

    # Save results
    results_file = f"evaluation_results_{len(all_results)}_samples.json"
    with open(results_file, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"📁 Results saved to {results_file}")

    return all_results


In [ ]:
#!pip install transformers==4.31.0
#pip install git+https://github.com/LAION-AI/CLAP.git


In [ ]:
#import laion_clap
evaluator = ObjectiveEvaluator()
midi_dir = "/content/drive/MyDrive/lmd_full"
results = batch_evaluate_midicaps(evaluator, num_samples=100, midi_dir=midi_dir)


/usr/local/lib/python3.11/dist-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4314.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Load our best checkpoint in the paper.
Download completed!
Load Checkpoint...
logit_scale_a 	 Loaded
logit_scale_t 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_real.weight 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_imag.weight 	 Loaded
audio_branch.logmel_extractor.melW 	 Loaded
audio_branch.bn0.weight 	 Loaded
audio_branch.bn0.bias 	 Loaded
audio_branch.patch_embed.proj.weight 	 Loaded
audio_branch.patch_embed.proj.bias 	 Loaded
audio_branch.patch_embed.norm.weight 	 Loaded
audio_branch.patch_embed.norm.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm1.weight 	 Loaded
audio_branch.layers.0.blocks.0.norm1.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.relative_position_bias_table 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm2.weight 	 Loaded
aud

README.md:   0%|          | 0.00/3.94k [00:00<?, ?B/s]

...

Final audio validation - type: <class 'numpy.ndarray'>, shape: (24976227,), dtype: float32
Saved temporary audio file: /tmp/tmptni10uaz.wav
Successfully used get_audio_embedding_from_filelist

Processing sample 99/100 (Original index: 85799)
Caption: A slow and dark electronic ambient song that evokes a sense of space through its melodic soundscapes...
Ground truth MIDI: /content/drive/MyDrive/lmd_full/c/c9711ca5973e8c523ac2e2d1a8a03f82.mid


100%|██████████| 2000/2000 [03:26<00:00,  9.67it/s]


Final audio validation - type: <class 'numpy.ndarray'>, shape: (12934084,), dtype: float32
Saved temporary audio file: /tmp/tmpllu30y3l.wav
Successfully used get_audio_embedding_from_filelist

Processing sample 100/100 (Original index: 90420)
Caption: A slow and melodic electronic song that evokes a sense of relaxation and meditation. Electric and ac...
Ground truth MIDI: /content/drive/MyDrive/lmd_full/c/cb05f1c25f80332000d0b3820e38856a.mid


100%|██████████| 2000/2000 [03:26<00:00,  9.67it/s]


Final audio validation - type: <class 'numpy.ndarray'>, shape: (2606274,), dtype: float32
Saved temporary audio file: /tmp/tmp_ute5uyp.wav
Successfully used get_audio_embedding_from_filelist

=== AGGREGATE RESULTS (100 samples) ===
Average Compression Ratio (Gen): 0.7522
Average Compression Ratio (GT): 2.8405
Average CLAP Similarity: 0.1405 ± 0.0637
Tempo Bin Accuracy (TB): 0.4200 | TBT: 0.7200
Correct Key Accuracy (CK): 0.2800 | CKD: 0.2800
📁 Results saved to evaluation_results_100_samples.json
